In [7]:
import json
import shutil
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split

In [8]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Projet :", PROJECT_ROOT)
print("Src    :", SRC_PATH)

Projet : c:\Users\chris\Desktop\ml-filrouge-project
Src    : c:\Users\chris\Desktop\ml-filrouge-project\src


In [9]:
df = pd.read_csv(PROJECT_ROOT / "data" / "df_preprocessed.csv")

X = df.drop(columns=["evo_conso_scaled"])
y = df["evo_conso_scaled"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

df.head()

,PRECIPITATION_QUANTITE_24H_scaled,TEMP_MIN_SOUS_ABRI_scaled,TEMP_MAX_SA_scaled,MOYENNE_TEMP_HORAIRES_SA_PONDEREE_scaled,DUREE_GEL_SOUSABRI_scaled,MOYENNE_FORCE_VENT_10M_scaled,MAX_FORCE_VENT_10M_scaled,DIRECTION_FORCE_VENT_scaled,MOYENNE_HUMIDITES_RELATIVES_HORAIRES_scaled,EPAISSEUR_NEIGE_scaled,...,13_encoded,83_encoded,84_encoded,Vendredi_encoded,Samedi_encoded,Dimanche_encoded,Lundi_encoded,Mardi_encoded,Mercredi_encoded,Jeudi_encoded
0,0.758159,-1.419764,-2.115097,-1.878023,5.714345,-0.869526,0.0,-3.667664e-16,-1.187787,4.837414,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.281063,-1.763619,-1.765881,-1.853167,4.382205,-0.869526,0.0,-3.667664e-16,-1.187787,4.926982,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.281063,-1.909096,-1.867266,-2.101727,4.693614,-0.869526,0.0,-3.667664e-16,-1.187787,4.479142,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.087693,-2.266177,-1.912326,-2.163867,5.420236,-0.869526,0.0,-3.667664e-16,-1.187787,4.031302,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,-0.281063,-2.160375,-2.205217,-2.226007,5.900325,-0.869526,0.0,-3.667664e-16,-1.187787,4.031302,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [16]:
def train_model(X_train, y_train):
    """Entraîne trois modèles de régression et retourne le meilleur
    basé sur la MAE en cross-validation."""
    model_v1 = {
        "RandomForest": RandomForestRegressor(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "LinearRegression": LinearRegression(),
        "GradientBoosting": GradientBoostingRegressor(random_state=42),
    }

    best_model = None
    best_score = -np.inf

    for _, model in model_v1.items():
        scores = cross_val_score(
            model, X_train, y_train, cv=5, scoring="neg_mean_absolute_error"
        )
        mean_score = scores.mean()
        if mean_score > best_score:
            best_score = mean_score
            best_model = model

    # Entraîne le meilleur modèle sur l'ensemble des données d'entraînement
    best_model.fit(X_train, y_train)
    print(f"Meilleur modèle sélectionné : {type(best_model).__name__}")
    return best_model

In [18]:
def evaluate(model, X_test, y_test):
    predictions = model.predict(X_test)
    return {
        "mae": float(mean_absolute_error(y_test, predictions)),
        "rmse": float(np.sqrt(mean_squared_error(y_test, predictions))),
        "r2": float(r2_score(y_test, predictions)),
    }


best_model = train_model(X_train, y_train)
metrics_v1 = evaluate(best_model, X_test, y_test)
metrics_v1

Meilleur modèle sélectionné : RandomForestRegressor


{'mae': 0.1497980582054019,
 'rmse': 0.27319093745923956,
 'r2': 0.9257425076898982}

In [20]:
models_dir = PROJECT_ROOT / "artifacts" / "models"
metrics_dir = PROJECT_ROOT / "artifacts" / "metrics"
models_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

# Sauvegarde du meilleur modèle
joblib.dump(best_model, models_dir / "model_v1.joblib")

# Sauvegarde des métriques
with open(metrics_dir / "metrics_v1.json", "w", encoding="utf-8") as file:
    json.dump(metrics_v1, file, indent=2)

print("Modèle V1 sauvegardé")

Modèle V1 sauvegardé


In [21]:
def train_model(X_train, y_train):
    """Entraîne trois modèles de régression et retourne le meilleur
    basé sur la MAE en cross-validation."""
    model_v2 = {
        "RandomForest": RandomForestRegressor(
            n_estimators=100, random_state=42, n_jobs=-1
        ),
        "LinearRegression": LinearRegression(),
        "GradientBoosting": GradientBoostingRegressor(random_state=42),
    }

    best_model = None
    best_score = -np.inf

    for _, model in model_v2.items():
        scores = cross_val_score(
            model, X_train, y_train, cv=5, scoring="neg_mean_absolute_error"
        )
        mean_score = scores.mean()
        if mean_score > best_score:
            best_score = mean_score
            best_model = model

    # Entraîne le meilleur modèle sur l'ensemble des données d'entraînement
    best_model.fit(X_train, y_train)
    print(f"Meilleur modèle sélectionné : {type(best_model).__name__}")
    return best_model

In [22]:
def evaluate(model, X_test, y_test):
    predictions2 = model.predict(X_test)
    return {
        "mae": float(mean_absolute_error(y_test, predictions2)),
        "rmse": float(np.sqrt(mean_squared_error(y_test, predictions2))),
        "r2": float(r2_score(y_test, predictions2)),
    }


best_model2 = train_model(X_train, y_train)
metrics_v2 = evaluate(best_model, X_test, y_test)
metrics_v2

Meilleur modèle sélectionné : RandomForestRegressor


{'mae': 0.1497980582054019,
 'rmse': 0.27319093745923956,
 'r2': 0.9257425076898982}

In [23]:
joblib.dump(best_model2, models_dir / "model_v2.joblib")

with open(metrics_dir / "metrics_v2.json", "w", encoding="utf-8") as file:
    json.dump(metrics_v2, file, indent=2)

print("Modèle V2 sauvegardé")

Modèle V2 sauvegardé


In [25]:
# Comparer les deux modèles et choisir le meilleur
print("\n" + "=" * 50)
print("      COMPARAISON DES MODÈLES V1 ET V2")
print("=" * 50)

print("\nModèle V1:")
print(f"  MAE  : {metrics_v1['mae']:.4f}")
print(f"  RMSE : {metrics_v1['rmse']:.4f}")
print(f"  R²   : {metrics_v1['r2']:.4f}")

print("\nModèle V2:")
print(f"  MAE  : {metrics_v2['mae']:.4f}")
print(f"  RMSE : {metrics_v2['rmse']:.4f}")
print(f"  R²   : {metrics_v2['r2']:.4f}")

# Choisir le meilleur modèle basé sur la MAE (plus basse = meilleur)
best_model_name = "v1" if metrics_v1["mae"] < metrics_v2["mae"] else "v2"
best_metrics = metrics_v1 if best_model_name == "v1" else metrics_v2
best_model_final = best_model if best_model_name == "v1" else best_model2

print("\n" + "=" * 50)
print(f"✓ Meilleur modèle: {best_model_name.upper()}")
print(f"  MAE  : {best_metrics['mae']:.4f}")
print(f"  RMSE : {best_metrics['rmse']:.4f}")
print(f"  R²   : {best_metrics['r2']:.4f}")
print("=" * 50)


      COMPARAISON DES MODÈLES V1 ET V2

Modèle V1:
  MAE  : 0.1498
  RMSE : 0.2732
  R²   : 0.9257

Modèle V2:
  MAE  : 0.1498
  RMSE : 0.2732
  R²   : 0.9257

✓ Meilleur modèle: V2
  MAE  : 0.1498
  RMSE : 0.2732
  R²   : 0.9257


In [27]:
# Déterminer le nom du fichier modèle à copier
model_filename = f"model_{best_model_name}.joblib"

# Copier le meilleur modèle vers model_latest.joblib
shutil.copy(
    models_dir / model_filename,
    models_dir / "model_latest.joblib",
)

# Copier les métriques correspondantes
metrics_filename = f"metrics_{best_model_name}.json"

with open(metrics_dir / metrics_filename, "r", encoding="utf-8") as src:
    latest_metrics = json.load(src)

with open(metrics_dir / "metrics_latest.json", "w", encoding="utf-8") as dst:
    json.dump(latest_metrics, dst, indent=2)

print("Modèle actif : model_latest.joblib")

Modèle actif : model_latest.joblib


In [ ]:
models = list((PROJECT_ROOT / "artifacts" / "models").iterdir())
metrics = list((PROJECT_ROOT / "artifacts" / "metrics").iterdir())
models, metrics

([WindowsPath('c:/Users/chris/Desktop/ml-filrouge-project/artifacts/models/model_latest.joblib'),
  WindowsPath('c:/Users/chris/Desktop/ml-filrouge-project/artifacts/models/model_v1.joblib'),
  WindowsPath('c:/Users/chris/Desktop/ml-filrouge-project/artifacts/models/model_v2.joblib')],
 [WindowsPath('c:/Users/chris/Desktop/ml-filrouge-project/artifacts/metrics/metrics_latest.json'),
  WindowsPath('c:/Users/chris/Desktop/ml-filrouge-project/artifacts/metrics/metrics_v1.json'),
  WindowsPath('c:/Users/chris/Desktop/ml-filrouge-project/artifacts/metrics/metrics_v2.json')])